# Agricultural Question Answering with a Fine-Tuned Small Language Model
### TinyLlama-1.1B + QLoRA — Notebook B: Reload & Question Answering

---


## Purpose of This Notebook

This notebook demonstrates that **the trained model persists across sessions and can be used
for question answering without requiring retraining**.

It is deliberately a **separate file from Notebook A** (which handled data preparation,
training, and evaluation) so that opening this notebook constitutes a genuinely new session —
a new Colab runtime, a new Python process, no variables carried over. This notebook contains:

- **No dataset download**
- **No model training code** (`Trainer`, `TrainingArguments`, `.train()` do not appear below)
- **No LoRA/QLoRA configuration step** (`LoraConfig`, `get_peft_model` do not appear below)

It only mounts the same Google Drive used by Notebook A, loads the tokenizer and the **already
trained** LoRA adapter directly from the files saved there, and answers questions with it.
If this notebook runs correctly on its own, that is the demonstration: the model's weights
were properly saved by Notebook A and are fully usable from disk alone.

**Prerequisite:** Notebook A must have been run at least once, so that
`/content/drive/MyDrive/Agriculture_QA/models/tinyllama_qlora_final` exists on Drive.

##OR
Copy the folder of name "Agriculture_QA" from my git repo and paste in MyDrive of your Google Drive

# Part B1: Minimal Environment Setup

Only what is needed to load and run the model — no dataset libraries beyond what `transformers`
itself needs.


In [2]:
!pip install -q --no-warn-conflicts \
transformers==4.56.2 \
peft==0.17.1 \
accelerate==1.10.1 \
bitsandbytes==0.47.0 \
sentencepiece

import json
import os
import re
import time

import torch
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

print("=" * 60)
print("Environment Summary")
print("=" * 60)
print("Torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.1/40.1 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.6/11.6 MB 109.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 504.9/504.9 kB 38.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 374.9/374.9 kB 32.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.3/61.3 MB 13.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 39.5 MB/s eta 0:00:00
Environment Summary
Torch: 2.11.0+cu128
CUDA available: True


In [3]:
from google.colab import drive
drive.mount('/content/drive')

project_path = "/content/drive/MyDrive/Agriculture_QA"
MODEL_NAME = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
final_model_path = project_path + "/models/tinyllama_qlora_final"
MAX_LENGTH = 512

assert os.path.isdir(final_model_path), (
    f"Could not find a saved adapter at {final_model_path}. "
    "Run Notebook A first to train and save the model."
)
print("Found saved adapter at:", final_model_path)
print("Contents:", os.listdir(final_model_path))


Mounted at /content/drive
Found saved adapter at: /content/drive/MyDrive/Agriculture_QA/models/tinyllama_qlora_final
Contents: ['domain_keywords.json', 'README.md', 'chat_template.jinja', 'adapter_config.json', 'tokenizer_config.json', 'special_tokens_map.json', 'tokenizer.model', 'tokenizer.json', 'adapter_model.safetensors']


# Part B2: Reload the Trained Model — No Training Involved

This is the persistence step: the base SML is loaded fresh, and the **already-trained** LoRA
adapter is attached to it by reading its saved weights from disk with `PeftModel.from_pretrained`.
Nothing here computes a gradient or calls `.train()`.


In [4]:
reload_start = time.time()

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
)

# Attaches the ALREADY-TRAINED adapter weights saved by Notebook A. This is
# the persistence step: loading, not training.
model = PeftModel.from_pretrained(base_model, final_model_path)
model.eval()

tokenizer = AutoTokenizer.from_pretrained(final_model_path)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

reload_time_sec = time.time() - reload_start
print(f"Model reloaded from saved files in {reload_time_sec:.2f} sec.")
print("No training was performed in this notebook.")


/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/608 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.20G [00:00<?, ?B/s]

/usr/local/lib/python3.13/dist-packages/bitsandbytes/backends/cuda/ops.py:212: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

Model reloaded from saved files in 32.58 sec.
No training was performed in this notebook.


# Part B3: Core Q&A Utilities

Same prompt template, generation logic, and confidence estimation as Notebook A's Part 6, so
answers are produced identically to how they were evaluated during training. `build_prompt`
takes only a question — no context/background text is required from the user.


In [5]:
def build_prompt(question, context=""):
    """Same template used in Notebook A. `context` defaults to empty, so a
    caller only needs to supply a question."""
    return f"""### Instruction:
Answer the agricultural question using the given context.

### Context:
{context}

### Question:
{question}

### Answer:
"""


In [6]:
import re

_SECTION_LEAK_PATTERN = re.compile(
    r"#{1,4}\s*(Question|Instruction|Context|Answer)\s*:", re.IGNORECASE
)


def generate_answer(prompt, gen_model=None, max_new_tokens=150):
    """Generate an answer plus a mean top-1 token-probability confidence
    score (same method as Notebook A)."""
    gen_model = model if gen_model is None else gen_model

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=MAX_LENGTH,
    ).to(gen_model.device)

    with torch.no_grad():
        output = gen_model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            num_beams=1,
            repetition_penalty=1.3,
            no_repeat_ngram_size=3,
            pad_token_id=tokenizer.eos_token_id,
            output_scores=True,
            return_dict_in_generate=True,
        )

    decoded = tokenizer.decode(output.sequences[0], skip_special_tokens=True)
    answer = decoded.split("### Answer:")[-1]

    # The model sometimes keeps generating past the answer and hallucinates a
    # new turn. It does not always repeat the exact "### Question:" format it
    # was trained on - it can drift to "##Question:", "#Question :", etc. This
    # regex catches 1-4 leading #s, optional space, the keyword, optional
    # space, then a colon, in any of those variants, and cuts there.
    leak_match = _SECTION_LEAK_PATTERN.search(answer)
    if leak_match:
        answer = answer[: leak_match.start()]
    answer = answer.strip()

    confidence = None
    if getattr(output, "scores", None):
        token_probs = [
            torch.max(torch.softmax(step_scores[0], dim=-1)).item()
            for step_scores in output.scores
        ]
        if token_probs:
            confidence = float(sum(token_probs) / len(token_probs))

    return answer, confidence


In [7]:
_FALLBACK_DOMAIN_KEYWORDS = {
    "crop", "crops", "soil", "fertilizer", "fertiliser", "irrigation", "seed",
    "seeds", "plant", "plants", "leaf", "leaves", "pest", "pests", "disease",
    "harvest", "farm", "farming", "farmer", "wheat", "maize", "rice", "corn",
    "soybean", "cotton", "tomato", "nitrogen", "pesticide", "herbicide",
    "drought", "weed", "weeds", "yield", "organic", "compost", "manure",
    "agriculture", "agricultural", "livestock", "greenhouse", "mulch",
    "drip", "nutrient", "nutrients", "pH", "grain",
}

# Loads the vocabulary Notebook A exported alongside the adapter, so the
# out-of-domain check works here without re-downloading the dataset. Falls
# back to a small manual list if that file isn't present.
_keywords_path = final_model_path + "/domain_keywords.json"
if os.path.exists(_keywords_path):
    with open(_keywords_path) as f:
        DOMAIN_KEYWORDS = set(json.load(f))
    print(f"Loaded {len(DOMAIN_KEYWORDS)} domain keywords from Notebook A's export.")
else:
    DOMAIN_KEYWORDS = _FALLBACK_DOMAIN_KEYWORDS
    print("domain_keywords.json not found - using fallback keyword list.")


def is_in_domain(question, min_overlap=1):
    words = set(re.findall(r"[a-zA-Z]{4,}", question.lower()))
    return len(words & DOMAIN_KEYWORDS) >= min_overlap


def confidence_label(confidence, high=0.60, medium=0.35):
    if confidence is None:
        return "Unknown"
    if confidence >= high:
        return "High"
    if confidence >= medium:
        return "Medium"
    return "Low"


Loaded 2438 domain keywords from Notebook A's export.


In [8]:
def ask(question, gen_model=None, max_new_tokens=150, retry_tokens=250):
    """End-to-end, question-only Q&A."""
    prompt = build_prompt(question)
    answer, confidence = generate_answer(prompt, gen_model=gen_model, max_new_tokens=max_new_tokens)

    if not answer:
        answer, confidence = generate_answer(prompt, gen_model=gen_model, max_new_tokens=retry_tokens)

    in_domain = is_in_domain(question)

    if not answer:
        answer = "I don't have enough information to answer that confidently."
        confidence = 0.0

    label = confidence_label(confidence)

    note = ""
    if not in_domain:
        note = (
            "This question doesn't closely match the agricultural training data — "
            "treat the answer as unsupported/low-confidence."
        )
        label = "Low" if label != "High" else label
    elif label == "Low":
        note = "The model was not very confident in this answer — verify independently."

    return {
        "question": question,
        "answer": answer,
        "confidence": confidence,
        "confidence_label": label,
        "in_domain": in_domain,
        "note": note,
    }


# Part B4: Persistence Proof

A fixed test question is asked here, in this brand-new session, using only the reloaded model.
If Notebook A's saved results are still on Drive, this cell also compares against the answer
recorded there for the same question — if it matches, that is direct, saved-to-disk evidence
that persistence held across the two separate notebooks/sessions.


In [9]:
PERSISTENCE_TEST_QUESTION = "What causes yellow leaves in maize?"

result = ask(PERSISTENCE_TEST_QUESTION)

print("=" * 60)
print("Answer produced in THIS (new) session, from the reloaded model")
print("=" * 60)
print("Question  :", result["question"])
print("Answer    :", result["answer"])
print("Confidence:", result["confidence_label"],
      f"({result['confidence']:.2f})" if result["confidence"] is not None else "")

persistence_check = {
    "question": PERSISTENCE_TEST_QUESTION,
    "answer_in_notebook_b": result["answer"],
    "reload_time_sec": reload_time_sec,
    "saved_model_path": final_model_path,
    "trained_in_this_notebook": False,
}

prior_record_path = project_path + "/results/persistence_validation.json"
if os.path.exists(prior_record_path):
    with open(prior_record_path) as f:
        prior_record = json.load(f)
    prior_answer = prior_record.get("answer_before_reset") or prior_record.get("answer_after_reload")
    if prior_answer:
        matches = prior_answer.strip() == result["answer"].strip()
        persistence_check["matches_prior_recorded_answer"] = matches
        print("\nCompared against a prior saved answer for the same question:")
        print("  Prior answer :", prior_answer)
        print("  MATCH" if matches else "  DIFFERS", "- " +
              ("identical output reproduced in a new session with no retraining."
               if matches else
               "outputs differ; if generation settings are unchanged this is worth investigating."))
else:
    print("\n(No prior persistence_validation.json found on Drive to compare against — "
          "this is still valid evidence that the model loaded and answered with zero "
          "training code in this notebook.)")

os.makedirs(project_path + "/results", exist_ok=True)
with open(project_path + "/results/notebook_b_persistence_check.json", "w") as f:
    json.dump(persistence_check, f, indent=2)
print("\nSaved to:", project_path + "/results/notebook_b_persistence_check.json")


/usr/local/lib/python3.13/dist-packages/bitsandbytes/backends/cuda/ops.py:464: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


Answer produced in THIS (new) session, from the reloaded model
Question  : What causes yellow leaves in maize?
Answer    : drought stress 

 ##

################

#### Excerpt from EPA-201C-35489-02B1 as part of the adjusted crop management sequence for corn that follows pollination, respectively:
a) Leaf and stem water use decreases due to drought stress   b) Corn silks become stiff or brittle    c) Yellow leaf spots appear     d) Swelling or death of kernels occurs      e) Pollen grains fail to develop into an embryo  

########
Confidence: High (0.63)

(No prior persistence_validation.json found on Drive to compare against — this is still valid evidence that the model loaded and answered with zero training code in this notebook.)

Saved to: /content/drive/MyDrive/Agriculture_QA/results/notebook_b_persistence_check.json


# Part B5: Interactive Question Answering

Type only a question — no context or background text is required.


In [10]:
print("=" * 60)
print("Interactive Agricultural Q&A  (Notebook B — reloaded model, no retraining)")
print("=" * 60)
print("Just type your question (or type 'exit' to stop):\n")

while True:
    question = input("Your question (or 'exit'): ").strip()

    if question.lower() == "exit":
        print("Session ended.")
        break

    if not question:
        print("\nPlease type a question.\n")
        continue

    result = ask(question)

    print("\n" + "-" * 60)
    print("Question  :", result["question"])
    print("Answer    :", result["answer"])
    conf_str = f'{result["confidence"]:.2f}' if result["confidence"] is not None else "N/A"
    print(f"Confidence: {result['confidence_label']} ({conf_str})")
    if result["note"]:
        print("Note      :", result["note"])
    print("-" * 60 + "\n")


Interactive Agricultural Q&A  (Notebook B — reloaded model, no retraining)
Just type your question (or type 'exit' to stop):

Your question (or 'exit'): What factors affect the cost of irrigation distribution equipment?

------------------------------------------------------------
Question  : What factors affect the cost of irrigation distribution equipment?
Answer    : the size and shape of the area to be irrigated 

 ##
Confidence: High (0.66)
------------------------------------------------------------

Your question (or 'exit'): What is 70 centibars equivalent to in terms of plant available water?

------------------------------------------------------------
Question  : What is 70 centibars equivalent to in terms of plant available water?
Answer    : 15 centibar or about one and a half inches  

 ##
Confidence: High (0.66)
------------------------------------------------------------

Your question (or 'exit'): How often should I check my irrigation system for leaks?

--------------